In [ ]:
!pip install -q torch transformers huggingface_hub numpy pandas networkx matplotlib scikit-learn

## Imports

In [ ]:
import pandas as pd
import numpy as np
import torch
import re
from sklearn.model_selection import train_test_split
from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, f1_score, classification_report

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import pandas as pd
import pickle
import re
from collections import defaultdict

PATH_TO_PKL = '/path/to/train_split.pkl'
PATH_TO_LABELED_CSV = '/path/to/llm_labeled_balanced_final.csv'
OUTPUT_PATH = '/path/to/llm_labeled_3turns_FINAL.csv'

def create_final_3turn_dataset():
    print("1. Loading Pickle...")
    with open(PATH_TO_PKL, 'rb') as f:
        train_data = pickle.load(f)

    # 2. Build a Map of {conv_id: {turn_num: text}}
    dialogue_map = defaultdict(dict)
    for item in train_data:
        cid = str(item['conversation_id']).replace('conv_', '')
        text = item['processed_text']

        # Extract Turn Number [Turn X]
        match = re.search(r'\[Turn (\d+)\]', text)
        if match:
            turn_num = int(match.group(1))
            dialogue_map[cid][turn_num] = text

    # 3. Load CSV
    df = pd.read_csv(PATH_TO_LABELED_CSV)
    enriched_rows = []

    def clean(t):
        if not t or t == "N/A": return "N/A"
        t = re.sub(r'\[Turn \d+\]', '', t)
        t = re.sub(r'\[SPEAKER\]', '', t)
        t = re.sub(r'\[/?E[12]\]', '', t)
        return ' '.join(t.split()).strip()

    print("2. Extracting Context...")
    for _, row in df.iterrows():
        cid = str(row['conv_id']).replace('conv_', '')

        # Get target turn number from the current sentence
        match = re.search(r'\[Turn (\d+)\]', row['sentence'])
        if match and cid in dialogue_map:
            target_num = int(match.group(1))

            # Get the actual turns using the numbers
            t_curr = dialogue_map[cid].get(target_num, "")
            t_prev = dialogue_map[cid].get(target_num - 1, "N/A")
            t_next = dialogue_map[cid].get(target_num + 1, "N/A")

            # Final Format Construction
            row['sentence'] = f"(-1 Turn: {clean(t_prev)}) (Target Turn: {clean(t_curr)}) (+1 Turn: {clean(t_next)})"
            enriched_rows.append(row)

    final_df = pd.DataFrame(enriched_rows)
    final_df.to_csv(OUTPUT_PATH, index=False)

    # Validation Print
    print(f"\n3. Success! Created {len(final_df)} rows.")
    print("Sample Check (Row 0):")
    print(final_df.iloc[0]['sentence'])
    return final_df

df = create_final_3turn_dataset()

1. Loading Pickle...
2. Extracting Context...

3. Success! Created 1179 rows.
Sample Check (Row 0):
(-1 Turn: Speaker 3: Okay, Ross is in the bathroom.) (Target Turn: Speaker 1: Oh my God, its happening. It's already started. I'm Kip.) (+1 Turn: Speaker 2: Hey, you're not Kip!)


## Context Extraction code

In [ ]:
df = pd.read_csv('/path/to/llm_labeled_3turns.csv')

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['llm_label'].value_counts()}")


Total samples: 1179
Label distribution:
llm_label
positive    400
neutral     393
negative    386
Name: count, dtype: int64


## Clean Sentences (Remove Entity Markers)


In [ ]:
def clean_sentence(text):
    """
    Remove entity markers [E1], [E2], [SPEAKER], [Turn X] for clean text.
    Keep only the actual dialogue content.
    """
    import re

    text = re.sub(r'\[Turn \d+\]', '', text)

    text = re.sub(r'\[SPEAKER\]', '', text)

    text = re.sub(r'\[/?E[12]\]', '', text)

    text = re.sub(r'Speaker \d+:', '', text)

    # Clean up extra whitespace
    text = ' '.join(text.split())

    return text.strip()

# Apply cleaning
df['sentence_clean'] = df['sentence'].apply(clean_sentence)
print("\nSample cleaned sentences:")
for i in range(3):
    print(f"\nOriginal: {df.iloc[i]['sentence'][:100]}...")
    print(f"Cleaned:  {df.iloc[i]['sentence_clean'][:100]}...")


Sample cleaned sentences:

Original: (-1 Turn: Speaker 1: Oh my God, its happening. It's already started. I'm Kip.) (Target Turn: Speaker...
Cleaned:  (-1 Turn: Oh my God, its happening. It's already started. I'm Kip.) (Target Turn: Oh my God, its hap...

Original: (-1 Turn: Speaker 3: Yeah, Ross can't go so it's between my friend Eric Prower who has breath issues...
Cleaned:  (-1 Turn: Yeah, Ross can't go so it's between my friend Eric Prower who has breath issues and Dan wi...

Original: (-1 Turn: Speaker 3: Umm, I'm sorry Judy, I couldn't find that bowl that you and Jack were looking f...
Cleaned:  (-1 Turn: Umm, I'm sorry Judy, I couldn't find that bowl that you and Jack were looking for.) (Targe...


## Map Labels to Integers


In [ ]:
label_map = {
    'positive': 0,
    'negative': 1,
    'neutral': 2
}

df['label'] = df['llm_label'].map(label_map)

print(f"\nLabel encoding:")
for label, idx in label_map.items():
    count = (df['label'] == idx).sum()
    print(f"  {label} ({idx}): {count} samples")


Label encoding:
  positive (0): 400 samples
  negative (1): 386 samples
  neutral (2): 393 samples


## Train/Val Split

In [ ]:
train_df, val_df = train_test_split(
    df[['sentence_clean', 'label', 'llm_confidence']],
    test_size=0.20,
    stratify=df['label'],
    random_state=42
)

print(f"\n✓ Train/Val split:")
print(f"  Train: {len(train_df)} samples")
print(f"  Val:   {len(val_df)} samples")

# Show distribution
print(f"\n✓ Train label distribution:")
print(train_df['label'].value_counts())
print(f"\n✓ Val label distribution:")
print(val_df['label'].value_counts())


✓ Train/Val split:
  Train: 943 samples
  Val:   236 samples

✓ Train label distribution:
label
0    320
2    314
1    309
Name: count, dtype: int64

✓ Val label distribution:
label
0    80
2    79
1    77
Name: count, dtype: int64


## Load RoBERTa-GoEmotions Model

In [ ]:
MODEL_NAME = "Lakssssshya/roberta-large-goemotions"

print(f"\nLoading {MODEL_NAME}...")

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    problem_type="single_label_classification",
    ignore_mismatched_sizes=True
)

print(f"✓ Model loaded: {model.num_parameters():,} parameters")


Loading Lakssssshya/roberta-large-goemotions...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at Lakssssshya/roberta-large-goemotions and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded: 355,362,819 parameters


## Tokenize Data

In [ ]:
def tokenize_function(examples):
    """Tokenize sentences."""
    return tokenizer(
        examples['sentence_clean'],
        truncation=True,
        padding='max_length',
        max_length=128  # Most DialogRE sentences are short
    )

# Convert to Hugging Face Dataset
train_dataset = Dataset.from_pandas(train_df[['sentence_clean', 'label']])
val_dataset = Dataset.from_pandas(val_df[['sentence_clean', 'label']])

# Tokenize
print("\nTokenizing datasets...")
train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)

# Set format for PyTorch
train_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])
val_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

print(f"✓ Tokenization complete")


Tokenizing datasets...


Map:   0%|          | 0/943 [00:00<?, ? examples/s]

Map:   0%|          | 0/236 [00:00<?, ? examples/s]

✓ Tokenization complete


## Define Metrics

In [ ]:
def compute_metrics(eval_pred):
    """Compute F1, accuracy, precision, recall."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)

    accuracy = accuracy_score(labels, predictions)
    f1_macro = f1_score(labels, predictions, average='macro', zero_division=0)

    # Per-class F1
    f1_per_class = f1_score(labels, predictions, average=None, zero_division=0)

    return {
        'accuracy': accuracy,
        'f1_macro': f1_macro,
        'f1_positive': f1_per_class[0],
        'f1_negative': f1_per_class[1],
        'f1_neutral': f1_per_class[2]
    }

## Training Configuration

In [ ]:
training_args = TrainingArguments(
    output_dir='/roberta-dialogre-finetuned',

    # Training hyperparameters
    num_train_epochs=15,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    weight_decay=0.01,
    warmup_steps=100,

    # Evaluation
    eval_strategy="steps",
    eval_steps=50,
    logging_steps=20,

    # Saving
    save_strategy="steps",
    save_steps=50,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,

    # Performance
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2,

    # Reproducibility
    seed=19,
    report_to=[]
)

print("\n✓ Training configuration set")


✓ Training configuration set


## Initialize Trainer

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=3,
            early_stopping_threshold=0.01
        )
    ]
)

print("✓ Trainer initialized")

✓ Trainer initialized


## Training

In [ ]:
print("\n" + "="*70)
print("STARTING TRAINING")
print("="*70)

train_result = trainer.train()

print("\n" + "="*70)
print("✓ TRAINING COMPLETE!")
print("="*70)
print(f"\nFinal training loss: {train_result.training_loss:.4f}")
print(f"Total training time: {train_result.metrics['train_runtime']:.1f}s")


STARTING TRAINING


Step,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Positive,F1 Negative,F1 Neutral
50,1.087900,1.070040,0.457627,0.374010,0.578313,0.471429,0.072289
100,0.923800,0.918875,0.618644,0.608598,0.525424,0.647887,0.652482
150,0.653200,0.879717,0.669492,0.661467,0.704545,0.681564,0.598291
200,0.433300,0.655152,0.771186,0.770356,0.792899,0.760000,0.758170
250,0.425800,0.685713,0.779661,0.775910,0.797619,0.809524,0.720588
300,0.235500,0.764122,0.809322,0.808586,0.827160,0.814815,0.783784
350,0.214700,0.927120,0.762712,0.761706,0.791946,0.770950,0.722222
400,0.212800,0.962275,0.809322,0.808830,0.844720,0.802548,0.779221
450,0.070500,1.147314,0.788136,0.786945,0.819876,0.785714,0.755245



✓ TRAINING COMPLETE!

Final training loss: 0.5003
Total training time: 684.4s


## Evaluate on Validation Set

In [ ]:
print("\n" + "="*70)
print("VALIDATION SET EVALUATION")
print("="*70)

val_results = trainer.evaluate(eval_dataset=val_dataset)

print(f"\n✓ Validation Results:")
print(f"  Accuracy:     {val_results['eval_accuracy']:.3f}")
print(f"  F1 Macro:     {val_results['eval_f1_macro']:.3f}")
print(f"  F1 Positive:  {val_results['eval_f1_positive']:.3f}")
print(f"  F1 Negative:  {val_results['eval_f1_negative']:.3f}")
print(f"  F1 Neutral:   {val_results['eval_f1_neutral']:.3f}")


VALIDATION SET EVALUATION



✓ Validation Results:
  Accuracy:     0.809
  F1 Macro:     0.809
  F1 Positive:  0.845
  F1 Negative:  0.803
  F1 Neutral:   0.779


## Save Fine-tuned Model

In [ ]:
output_dir = './roberta-large-dialogre-finetuned'
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print(f"\n✓ Model saved to: {output_dir}")


✓ Model saved to: ./roberta-large-dialogre-finetuned


## Test on 150 Manual Annotations (Gold Standard)

In [ ]:
print("\n" + "="*70)
print("FINAL TEST ON 150 MANUAL ANNOTATIONS")
print("="*70)

# Load your 150 manual annotations
test_manual_df = pd.read_csv('/path/to/annotation_samples_150.csv')

# Clean sentences (if they have entity markers)
test_manual_df['sentence_clean'] = test_manual_df['sentence'].apply(clean_sentence)

# Map labels
test_manual_df['label'] = test_manual_df['manual_label'].map(label_map)

# Convert to dataset
test_dataset = Dataset.from_pandas(test_manual_df[['sentence_clean', 'label']])
test_dataset = test_dataset.map(tokenize_function, batched=True)
test_dataset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'label'])

# Evaluate
test_results = trainer.evaluate(eval_dataset=test_dataset)

print(f"\n✓ Test Results (150 Manual Annotations):")
print(f"  Accuracy:     {test_results['eval_accuracy']:.3f}")
print(f"  F1 Macro:     {test_results['eval_f1_macro']:.3f}")
print(f"  F1 Positive:  {test_results['eval_f1_positive']:.3f}")
print(f"  F1 Negative:  {test_results['eval_f1_negative']:.3f}")
print(f"  F1 Neutral:   {test_results['eval_f1_neutral']:.3f}")

# Get detailed predictions
test_predictions = trainer.predict(test_dataset)
test_preds = np.argmax(test_predictions.predictions, axis=1)
test_labels = test_predictions.label_ids

# Classification report
print("\n✓ Detailed Classification Report:")
print(classification_report(
    test_labels,
    test_preds,
    target_names=['positive', 'negative', 'neutral'],
    digits=3
))

# Confusion matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(test_labels, test_preds)

print("\n✓ Confusion Matrix:")
print("                 Predicted")
print("               Pos    Neg    Neu")
print(f"Actual Pos   {cm[0][0]:5d}  {cm[0][1]:5d}  {cm[0][2]:5d}")
print(f"       Neg   {cm[1][0]:5d}  {cm[1][1]:5d}  {cm[1][2]:5d}")
print(f"       Neu   {cm[2][0]:5d}  {cm[2][1]:5d}  {cm[2][2]:5d}")


FINAL TEST ON 150 MANUAL ANNOTATIONS


Map:   0%|          | 0/150 [00:00<?, ? examples/s]


✓ Test Results (150 Manual Annotations):
  Accuracy:     0.633
  F1 Macro:     0.635
  F1 Positive:  0.630
  F1 Negative:  0.714
  F1 Neutral:   0.562

✓ Detailed Classification Report:
              precision    recall  f1-score   support

    positive      0.702     0.571     0.630        70
    negative      0.750     0.682     0.714        44
     neutral      0.472     0.694     0.562        36

    accuracy                          0.633       150
   macro avg      0.641     0.649     0.635       150
weighted avg      0.661     0.633     0.638       150


✓ Confusion Matrix:
                 Predicted
               Pos    Neg    Neu
Actual Pos      40      9     21
       Neg       7     30      7
       Neu      10      1     25


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score
from sklearn.utils import resample

def run_bootstrapping_analysis(y_true, y_pred, model_name="Model", n_iterations=1000):
    """
    Calculates 95% Confidence Intervals for Macro F1.
    """
    bootstrapped_f1s = []
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    print(f"Bootstrapping {model_name}...")
    for i in range(n_iterations):
        indices = resample(np.arange(len(y_true)), replace=True)
        resampled_true = y_true[indices]
        resampled_pred = y_pred[indices]

        score = f1_score(resampled_true, resampled_pred, average='macro')
        bootstrapped_f1s.append(score)

    lower = np.percentile(bootstrapped_f1s, 2.5)
    upper = np.percentile(bootstrapped_f1s, 97.5)
    median = np.percentile(bootstrapped_f1s, 50)

    print(f"{model_name} Results:")
    print(f"  Median F1: {median:.3f}")
    print(f"  95% CI:    [{lower:.3f}, {upper:.3f}]")

    return bootstrapped_f1s, (lower, upper)

results_dist, ci_bounds = run_bootstrapping_analysis(test_labels, test_preds, "RoBERTa-3T")

Bootstrapping RoBERTa-3T...
RoBERTa-3T Results:
  Median F1: 0.631
  95% CI:    [0.560, 0.708]


### Save Results

In [ ]:
test_manual_df['roberta_pred'] = [['positive', 'negative', 'neutral'][p] for p in test_preds]
test_manual_df['correct'] = test_manual_df['label'] == test_preds

test_manual_df.to_csv('test_predictions_150_manual.csv', index=False)

print("\n✓ Saved: test_predictions_150_manual.csv")

# Save metrics
import json
metrics = {
    'validation': {
        'accuracy': val_results['eval_accuracy'],
        'f1_macro': val_results['eval_f1_macro'],
        'f1_positive': val_results['eval_f1_positive'],
        'f1_negative': val_results['eval_f1_negative'],
        'f1_neutral': val_results['eval_f1_neutral']
    },
    'test_150_manual': {
        'accuracy': test_results['eval_accuracy'],
        'f1_macro': test_results['eval_f1_macro'],
        'f1_positive': test_results['eval_f1_positive'],
        'f1_negative': test_results['eval_f1_negative'],
        'f1_neutral': test_results['eval_f1_neutral']
    },
    'train_samples': len(train_df),
    'val_samples': len(val_df),
    'test_samples': len(test_manual_df)
}

with open('finetuned_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print("✓ Saved: finetuned_metrics.json")


✓ Saved: test_predictions_150_manual.csv
✓ Saved: finetuned_metrics.json
